In [1]:
import pandas as pd
import numpy as np
import re

ANCHOR_MATRIX = "human_matrix_answered_9users.csv"
OPTIONS = set(list("ABCDE"))

def normalize_to_letter(x):
    if pd.isna(x): 
        return None
    s = str(x).strip().upper()
    m = re.search(r"\b([A-E])\b", s)
    return m.group(1) if m else None

def parse_cell(cell):
    """
    Returns list of letters from a cell like 'D', 'D|D', 'E|D', ''.
    """
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if not s:
        return []
    parts = [p.strip() for p in s.split("|")]
    letters = [normalize_to_letter(p) for p in parts]
    return [l for l in letters if l in OPTIONS]

# ---- load matrix ----
hm = pd.read_csv(ANCHOR_MATRIX)

# recover question_id index
if "question_id" in hm.columns:
    hm = hm.set_index("question_id")
elif "Unnamed: 0" in hm.columns:
    hm = hm.rename(columns={"Unnamed: 0": "question_id"}).set_index("question_id")
else:
    hm = hm.set_index(hm.columns[0])
    hm.index.name = "question_id"

hm.index = hm.index.astype(int)

# user columns = everything that is not counts
meta_cols = [c for c in hm.columns if c.startswith("n_") or c.startswith("count_")]
user_cols = [c for c in hm.columns if c not in meta_cols]

# ---- build repeated-pair records ----
records = []
for qid, row in hm.iterrows():
    for user in user_cols:
        letters = parse_cell(row[user])
        if len(letters) >= 2:  # repeated item for that user
            all_same = (len(set(letters)) == 1)
            records.append({
                "question_id": qid,
                "user": user,
                "answers": "|".join(letters),
                "n_attempts": len(letters),
                "consistent": all_same
            })

rep = pd.DataFrame(records)

# If there are no repeats, you'll see empty tables
if rep.empty:
    print("No repeated anchor responses found (no cells with multiple answers).")
else:
    # ---- TABLE 1: per-user consistency ----
    per_user = (rep.groupby("user", as_index=False)
                .agg(n_repeated_items=("consistent", "size"),
                     n_consistent=("consistent", "sum")))
    per_user["consistency_rate"] = per_user["n_consistent"] / per_user["n_repeated_items"]

    # ---- TABLE 2: per-question consistency ----
    per_question = (rep.groupby("question_id", as_index=False)
                    .agg(n_repeated_users=("consistent", "size"),
                         n_consistent=("consistent", "sum")))
    per_question["consistency_rate"] = per_question["n_consistent"] / per_question["n_repeated_users"]

    # ---- overall ----
    overall_rate = rep["consistent"].mean()

    print("Overall anchor-repeat consistency:", overall_rate)
    display(per_user.sort_values("consistency_rate", ascending=False))
    display(per_question.sort_values("consistency_rate", ascending=False))

    # optional: save
    # per_user.to_csv("anchor_repeat_consistency_per_user.csv", index=False)
    # per_question.to_csv("anchor_repeat_consistency_per_question.csv", index=False)


Overall anchor-repeat consistency: 0.8333333333333334


,user,n_repeated_items,n_consistent,consistency_rate
2,claudio (1),2,2,1.0
7,simone (8),2,2,1.0
4,eyildiz (3),2,2,1.0
6,patrizio (7),2,2,1.0
5,marina_andric (5),2,2,1.0
8,tbailoni (9),2,2,1.0
0,Gianlu94 (4),2,1,0.5
3,dragoni (2),2,1,0.5
1,MonicaConsolandi (6),2,1,0.5


,question_id,n_repeated_users,n_consistent,consistency_rate
0,2,1,1,1.0
1,9,2,2,1.0
3,34,2,2,1.0
5,53,1,1,1.0
11,136,2,2,1.0
6,62,1,1,1.0
7,79,2,2,1.0
8,83,1,1,1.0
10,114,1,1,1.0
2,32,2,1,0.5
